In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Setup

## install requirements

In [2]:
# DONE IIN MANAGE DEPENDENCIES

## Imports

In [3]:
import os
import gc
import json
import random

import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics.pairwise import cosine_similarity

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

from sentence_transformers import SentenceTransformer

from gensim.models import Word2Vec

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline,
    DataCollatorWithPadding
)

import wandb

from kaggle_secrets import UserSecretsClient

from huggingface_hub import login, whoami

import string

from transformers import get_linear_schedule_with_warmup

from torch.optim import AdamW



## Seeding for Reproducibility

In [4]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## Weights and Biases Setup

In [5]:
user_secrets = UserSecretsClient()
wandb_api_key = user_secrets.get_secret("WANB_API_KEY")


wandb.login(
    key=wandb_api_key
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: siddhartha12122002 (24f1002705-IITM) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## HuggingFace Setup (for Deploying)

In [6]:
hf_token = user_secrets.get_secret(
    "hf_token"
)

login(hf_token)

whoami()

{'type': 'user',
 'id': '69149e0c8c395b26b9d30cb6',
 'name': 'SiddharthaVarma12',
 'fullname': 'Siddhartha Varma',
 'canPay': False,
 'billingMode': 'prepaid',
 'periodEnd': 1782864000,
 'isPro': False,
 'avatarUrl': '/avatars/54e0ed6c83bbd2e679fbe09fd23af9c3.svg',
 'orgs': [],
 'auth': {'type': 'access_token',
  'accessToken': {'displayName': 'NPPE2',
   'role': 'fineGrained',
   'createdAt': '2025-12-15T10:22:57.136Z',
   'fineGrained': {'canReadGatedRepos': True,
    'global': ['discussion.write', 'post.write'],
    'scoped': [{'entity': {'_id': '69149e0c8c395b26b9d30cb6',
       'type': 'user',
       'name': 'SiddharthaVarma12'},
      'permissions': ['repo.content.read',
       'repo.write',
       'inference.serverless.write',
       'inference.endpoints.infer.write',
       'inference.endpoints.write',
       'user.webhooks.read',
       'user.webhooks.write',
       'collection.read',
       'collection.write',
       'discussion.write',
       'user.billing.read',
       'job

## MAP@3 function

In [7]:
# takes two inputs, one is a list of all correct answers, the other a list of corresponding predictions
def map_at_3(y_true, y_pred):
    score = 0.0

    if len(y_true) != len(y_pred):
        raise ValueError("Length mismatch")

    for i in range(len(y_true)):
        pred = y_pred[i]
        true = y_true[i]
        if true in pred:
            for j in range(len(pred)):
                if pred[j] == true:
                    score += 1/(j+1)
                    break
                
    return score/len(y_true)

## Data cleaning

In [8]:
import re

def clean_text(text):

    text = str(text).lower()

    text = re.sub(
        r"[^a-z0-9 ]",
        " ",
        text
    )

    return text

## Exploratory Data Analysis

In [9]:
df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
df

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A
...,...,...,...,...,...,...,...,...
1995,1996,What is the piezoelectric strain coefficient f...,d = 1.9·10‑12 m/V,d = 3.1·10‑12 m/V,d = 4.2·10‑12 m/V,d = 2.5·10‑12 m/V,d = 5.8·10‑12 m/V,B
1996,1997,Identify the correct statement: What is the sy...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,E
1997,1998,Determine the correct option: What does Earnsh...,A collection of point charges can be maintaine...,A collection of point charges can be maintaine...,A collection of point charges can be maintaine...,A collection of point charges cannot be mainta...,A collection of point charges can be maintaine...,D
1998,1999,Identify the correct statement: What is the re...,The atmosphere is a mechanism that is only inf...,"The atmosphere possesses both chaos and order,...",The atmosphere is a structure that is only inf...,The atmosphere is a completely chaotic mechani...,The atmosphere is a completely ordered structu...,B


In [10]:
df_test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
df_test

,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...
...,...,...,...,...,...,...,...
495,496,What are the constituents of cold dark matter?,"They are unknown, but possibilities include la...",They are known to be black holes and Preon stars.,They are only MACHOs.,They are clusters of brown dwarfs.,They are new particles such as RAMBOs.
496,497,Pick the best possible answer: What is a plane...,A framework of planets that are all located in...,A mechanism of planets that are all the same s...,Any set of gravitationally bound non-stellar o...,A mechanism of planets that are all located in...,A structure of planets that are all made of gas.
497,498,Pick the best possible answer: What is magneti...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...
498,499,Determine the correct option: What is the evid...,The Milky Way galaxy has a supermassive black ...,The Milky Way galaxy has a supermassive black ...,The Milky Way galaxy has a supermassive black ...,The Milky Way galaxy has a supermassive black ...,The star S2 follows an elliptical orbit with a...


# Milestone-4 (DaBERTa)

## setup

In [11]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)
print(torch.cuda.get_device_name(0))

cuda
Tesla T4


In [12]:
# Train/Validation split
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['answer'],
    random_state=42
)

print(len(train_df))
print(len(val_df))

1600
400


In [13]:
def expand_dataset(df):
    samples = []

    for _, row in df.iterrows():
        
        for option in ["A","B","C","D","E"]:
            samples.append({
                'prompt': row['prompt'],

                'option': row[option],
                
                "label": 1 if option == row["answer"] else 0,
    
                "question_id": row["id"]
            })

    return pd.DataFrame(samples)

In [14]:
train_pairs = expand_dataset(train_df)

val_pairs = expand_dataset(val_df)

In [15]:
train_pairs.head()

,prompt,option,label,question_id
0,Select the most accurate option: What is Modif...,MOND is a principle that explains the behavior...,0,387
1,Select the most accurate option: What is Modif...,MOND is a hypothesis that proposes a modificat...,0,387
2,Select the most accurate option: What is Modif...,MOND is a hypothesis that proposes a modificat...,1,387
3,Select the most accurate option: What is Modif...,MOND is a hypothesis that proposes a modificat...,0,387
4,Select the most accurate option: What is Modif...,MOND is a concept that explains the behavior o...,0,387


In [16]:
# Load Deberta 
MODEL_NAME = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

In [17]:
# create MCQDATASET class
class MCQDataset(Dataset):
    def __init__(self, dataframe):
        self.prompts = dataframe['prompt'].tolist()
        self.options = dataframe['option'].tolist()
        self.labels = dataframe['label'].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        encoding = tokenizer(
            self.prompts[idx],
            self.options[idx],
            truncation=True,
            max_length=256
        )

        return {
            "input_ids":
                encoding['input_ids'],

            "attention_mask":
                encoding['attention_mask'],

            "labels": int(self.labels[idx])
        }

In [18]:
train_dataset = MCQDataset(train_pairs)
val_dataset = MCQDataset(val_pairs)

In [19]:
# data collator
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt"
)

In [20]:
train_loader =  DataLoader(
    train_dataset,
    batch_size = 16,
    shuffle=True,
    collate_fn=data_collator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=data_collator
)

In [21]:
sample = train_dataset[0]

print(sample.keys())

print(type(sample["input_ids"]))

print(len(sample["input_ids"]))

print(sample["labels"])

dict_keys(['input_ids', 'attention_mask', 'labels'])
<class 'list'>
67
0


In [22]:
batch = next(iter(train_loader))

print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["labels"].shape)

torch.Size([16, 85])
torch.Size([16, 85])
torch.Size([16])


In [23]:
print(

    tokenizer.decode(

        sample["input_ids"],

        skip_special_tokens=False

    )

)

Select the most accurate option: What is Modified Newtonian Dynamics (MOND)? from the following choices. MOND is a principle that explains the behavior of light in the presence of strong gravitational fields. It is an alternative to the hypothesis of dark matter in terms of explaining why galaxies do not appear to obey the currently understood laws of physics.


## LoadModel

In [24]:
model = AutoModelForSequenceClassification.from_pretrained(

    MODEL_NAME,

    num_labels=2,

    torch_dtype=torch.float32

)

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias        

In [25]:
model = model.to(device)
next(model.parameters()).device

device(type='cuda', index=0)

### Optimizer

In [26]:
# Optimizer
optimizer = AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

In [27]:
# Learning rate Scheduler
epochs = 3

total_steps = len(train_loader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

In [28]:
batch = next(iter(train_loader))

print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["labels"].shape)

torch.Size([16, 109])
torch.Size([16, 109])
torch.Size([16])


In [29]:
encoding = tokenizer(
    train_pairs.iloc[0]["prompt"],
    train_pairs.iloc[0]["option"],
    truncation=True,
    padding="max_length",
    max_length=256,
    return_tensors="pt"
)

print(type(encoding["input_ids"]))
print(encoding["input_ids"].shape)

<class 'torch.Tensor'>
torch.Size([1, 256])


In [30]:
print(next(model.parameters()).dtype)
print(torch.isnan(next(model.parameters())).any())

torch.float32
tensor(False, device='cuda:0')


## Training Loop

### Start wandb run

In [31]:
import wandb

wandb.init(

    project="24f1002705-DLGenAI-t12026",

    name="deberta-v3-base",

    config={

        "model": MODEL_NAME,
        "epochs": epochs,
        "batch_size": 16,
        "learning_rate": 2e-5,
        "max_length": 256,
        "optimizer": "AdamW",
        "scheduler": "Linear Warmup"

    }

)

wandb: setting up run lyasli48
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260628_143629-lyasli48
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run deberta-v3-base
wandb: ⭐️ View project at https://wandb.ai/24f1002705-IITM/24f1002705-DLGenAI-t12026
wandb: 🚀 View run at https://wandb.ai/24f1002705-IITM/24f1002705-DLGenAI-t12026/runs/lyasli48


In [32]:
best_accuracy = 0

for epoch in range(epochs):

    ##################################
    # Training
    ##################################

    model.train()

    train_loss = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)

        attention_mask = batch["attention_mask"].to(device)

        labels = batch["labels"].long().to(device)

        optimizer.zero_grad()

        outputs = model(

            input_ids=input_ids,

            attention_mask=attention_mask,

            labels=labels

        )

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        scheduler.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    ##################################
    # Validation
    ##################################

    model.eval()

    val_loss = 0

    correct = 0

    total = 0

    with torch.no_grad():

        for batch in val_loader:

            input_ids = batch["input_ids"].to(device)

            attention_mask = batch["attention_mask"].to(device)

            labels = batch["labels"].long().to(device)

            outputs = model(

                input_ids=input_ids,

                attention_mask=attention_mask,

                labels=labels

            )

            val_loss += outputs.loss.item()

            predictions = torch.argmax(
                outputs.logits,
                dim=1
            )

            correct += (predictions == labels).sum().item()

            total += labels.size(0)

    val_loss /= len(val_loader)

    accuracy = correct / total

    ##################################
    # W&B Logging
    ##################################

    wandb.log({

        "epoch": epoch + 1,

        "train_loss": train_loss,

        "validation_loss": val_loss,

        "validation_accuracy": accuracy,

        "learning_rate": scheduler.get_last_lr()[0]

    })

    ##################################
    # Console
    ##################################

    print(

        f"Epoch {epoch+1}/{epochs}"

        f" | Train Loss: {train_loss:.4f}"

        f" | Val Loss: {val_loss:.4f}"

        f" | Accuracy: {accuracy:.4f}"

    )

    ##################################
    # Save Best Model
    ##################################

    if accuracy >= best_accuracy:

        best_accuracy = accuracy

        model.save_pretrained("best_model")

        tokenizer.save_pretrained("best_model")

        artifact = wandb.Artifact(

            "best-deberta-model",

            type="model"

        )

        artifact.add_dir("best_model")

        wandb.log_artifact(artifact)

print(f"\nBest Validation Accuracy: {best_accuracy:.4f}")

wandb.finish()

Epoch 1/3 | Train Loss: 0.5073 | Val Loss: 0.5008 | Accuracy: 0.8000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

wandb: Adding directory to artifact (best_model)... Done. 2.4s
wandb: WARNING Artifact "best-deberta-model" already exists with the same content. No new version will be created.


Epoch 2/3 | Train Loss: 0.5006 | Val Loss: 0.4884 | Accuracy: 0.8000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Epoch 3/3 | Train Loss: 0.4790 | Val Loss: 0.4341 | Accuracy: 0.8000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

wandb: Adding directory to artifact (best_model)... Done. 2.5s
wandb: uploading artifact best-deberta-model; updating run metadata



Best Validation Accuracy: 0.8000


wandb: uploading artifact best-deberta-model
wandb: 
wandb: Run history:
wandb:               epoch ▁▅█
wandb:       learning_rate █▅▁
wandb:          train_loss █▆▁
wandb: validation_accuracy ▁▁▁
wandb:     validation_loss █▇▁
wandb: 
wandb: Run summary:
wandb:               epoch 3
wandb:       learning_rate 0
wandb:          train_loss 0.47896
wandb: validation_accuracy 0.8
wandb:     validation_loss 0.43408
wandb: 
wandb: 🚀 View run deberta-v3-base at: https://wandb.ai/24f1002705-IITM/24f1002705-DLGenAI-t12026/runs/lyasli48
wandb: ⭐️ View project at: https://wandb.ai/24f1002705-IITM/24f1002705-DLGenAI-t12026
wandb: Synced 5 W&B file(s), 0 media file(s), 10 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260628_143629-lyasli48/logs


In [33]:
print(next(model.parameters()).dtype)
print(next(model.parameters()).device)

torch.float32
cuda:0


In [34]:
print(torch.isnan(next(model.parameters())).any())

tensor(False, device='cuda:0')


In [35]:
batch = next(iter(train_loader))

print(batch["input_ids"].dtype)
print(batch["attention_mask"].dtype)
print(batch["labels"].dtype)

torch.int64
torch.int64
torch.int64


In [36]:
import torch
import torch.nn.functional as F

OPTIONS = ["A", "B", "C", "D", "E"]

def predict_mcq(model, tokenizer, row, device):

    model.eval()

    scores = []

    with torch.no_grad():

        for option in OPTIONS:

            encoding = tokenizer(

                row["prompt"],
                row[option],

                truncation=True,
                padding="max_length",
                max_length=256,

                return_tensors="pt"

            )

            input_ids = encoding["input_ids"].to(device)
            attention_mask = encoding["attention_mask"].to(device)

            outputs = model(

                input_ids=input_ids,
                attention_mask=attention_mask

            )

            probabilities = F.softmax(outputs.logits, dim=1)

            score = probabilities[0,1].item()

            scores.append(score)

    ranking = [

        option

        for _, option in sorted(

            zip(scores, OPTIONS),

            reverse=True

        )

    ]

    return ranking, scores

In [37]:
row = val_df.iloc[0]

ranking, scores = predict_mcq(
    model,
    tokenizer,
    row,
    device
)

print("Prompt:\n")
print(row["prompt"])

print("\nTrue:", row["answer"])

print("\nRanking:", ranking)

print("\nScores:")

for option, score in zip(OPTIONS, scores):

    print(option, score)

Prompt:

What is a phageome?

True: C

Ranking: ['C', 'E', 'D', 'B', 'A']

Scores:
A 0.08247010409832001
B 0.0926523357629776
C 0.15513810515403748
D 0.09267043322324753
E 0.10370434075593948


In [38]:
predictions = []

for _, row in df_test.iterrows():

    ranking, _ = predict_mcq(

        model,
        tokenizer,
        row,
        device

    )

    predictions.append(

        " ".join(ranking[:3])

    )

In [39]:
y_pred = pd.DataFrame({
    'ID': df_test['id'],
    'Prediction': [' '.join(pred) for pred in predictions]
})

y_pred.to_csv('submission_DaBERTa_v1.csv', index=False)